In [1]:
# !pip uninstall gensim --yes
# !pip install gensim==3.8.1
# !pip install deeprobust==0.2.9
# !pip install torch_geometric
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.2.0+cpu.html

In [1]:
from deeprobust.graph.data import Dpr2Pyg, Pyg2Dpr
from deeprobust.graph.data import Dataset as DRDataset
import torch
from torch_geometric.data import Data
import numpy as np
from deeprobust.graph.data import Dataset, PrePtbDataset, PtbDataset
from deeprobust.graph.defense import GCN, RGCN, ProGNN, SimPGCN, GCNSVD, GCNJaccard
from deeprobust.graph.global_attack import Metattack, DICE, Random, PGDAttack
from scipy.sparse import csr_matrix
import torch
from torch.nn import Linear
import torch.nn.functional as F
from torch_geometric.transforms import Compose
from torch_geometric.datasets import Amazon
from torch_geometric.transforms.random_node_split import RandomNodeSplit
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.nn import GCNConv
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv
from sklearn.metrics import roc_auc_score

from torch_geometric.utils import negative_sampling
from torch_geometric.utils import train_test_split_edges

from copy import deepcopy
import torch.nn as nn
from IPython.display import Javascript  # Restrict height of output cell.
import matplotlib.pyplot as plt

from GSage import GSAGE
from GSaint import GSAINT
from GAT import GAT

In [3]:
seed = 15
ptb_rates = [0.05, 0.1, 0.15, 0.20, 0.25, 0.3]
# ptb_rates = [0.025, 0.05]
# ptb_rate = 0.25
# ptb_rate1 = 0.15
dataset = 'cora_ml'
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

In [5]:
data = DRDataset(root='/tmp/', name=dataset, setting='prognn')
adj, features, labels = data.adj, data.features, data.labels
idx_train, idx_val, idx_test = data.idx_train, data.idx_val, data.idx_test
idx_unlabeled = np.union1d(idx_val, idx_test)
idx_unlabeled = np.union1d(idx_val, idx_test)


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# budget = int(ptb_rate * (adj.todense().sum() // 2))
# print(budget)
# budget1 = int(ptb_rate1 * (adj.todense().sum() // 2))
# print(budget1)

Loading cora_ml dataset...
Selecting 1 largest connected components


In [6]:
np.save(f'/tmp/{dataset}_adj.npy',adj.todense())

In [7]:
np.save(f'/tmp/{dataset}_features.npy',features.todense())

In [8]:
np.save(f'/tmp/{dataset}_labels.npy',labels)

In [9]:
np.save(f'/tmp/{dataset}_idx_test',idx_test)

# Metattack

## GAT

In [10]:
surrogate5 = GAT(nfeat=features.shape[1],
      nhid=8, heads=8,
      nclass=labels.max().item() + 1,
      dropout=0.5, device=device)
surrogate5 = surrogate5.to(device)

pyg_data = Dpr2Pyg(data)
surrogate5.fit(pyg_data, verbose=True) # train with earlystopping
surrogate5.test()

Processing...
/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/deeprobust/graph/data/pyg_dataset.py:48: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:245.)
  edge_index = torch.LongTensor(dpr_data.adj.nonzero())
Done!
/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/torch_geometric/data/in_memory_dataset.py:157: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


=== training GAT model ===
Epoch 0, training loss: 1.9399274587631226
Epoch 10, training loss: 1.0134764909744263
Epoch 20, training loss: 0.5818580389022827
Epoch 30, training loss: 0.4689496159553528
Epoch 40, training loss: 0.42920982837677
Epoch 50, training loss: 0.46331340074539185
Epoch 60, training loss: 0.43876269459724426
Epoch 70, training loss: 0.3369770050048828
Epoch 80, training loss: 0.36580222845077515
Epoch 90, training loss: 0.3372045159339905
Epoch 100, training loss: 0.341568261384964
Epoch 110, training loss: 0.3200167119503021
Epoch 120, training loss: 0.24539759755134583
Epoch 130, training loss: 0.3510321378707886
Epoch 140, training loss: 0.3358899652957916
Epoch 150, training loss: 0.3447478711605072
Epoch 160, training loss: 0.3596220314502716
Epoch 170, training loss: 0.328563928604126
Epoch 180, training loss: 0.3852740228176117
Epoch 190, training loss: 0.28614532947540283
Epoch 200, training loss: 0.250931978225708
Epoch 210, training loss: 0.32926928997

0.8505338078291814

In [11]:
surrogate5.eval()
preds=surrogate5.predict()
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total
benchmark_clean = test_accuracy

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.8505338078291815


In [12]:
from copy import deepcopy

# ptb_rates = [0.1, 0.2, 0.3, 0.4, 0.5]
gat_results = []

for ptb in ptb_rates:
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate5, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)

  data_copied = deepcopy(data)
  data_copied.adj = modified_adj

  atk_model = GAT(nfeat=features.shape[1],
        nhid=8, heads=8,
        nclass=labels.max().item() + 1,
        dropout=0.5, device=device)
  atk_model = surrogate5.to(device)
  atk_model.fit(Dpr2Pyg(data_copied), patience=100, verbose=True)

  atk_acc = atk_model.test()

  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  gat_results.append(atk_acc - benchmark_clean)

Perturbing graph:   0%|          | 0/399 [00:00<?, ?it/s]

GCN loss on unlabled data: 1.5243231058120728
GCN acc on unlabled data: 0.47193675889328063
attack loss: 1.513676643371582


Perturbing graph:   0%|          | 1/399 [00:01<10:14,  1.54s/it]

GCN loss on unlabled data: 1.523925542831421
GCN acc on unlabled data: 0.5280632411067193
attack loss: 1.5214112997055054


Perturbing graph:   1%|          | 2/399 [00:02<09:29,  1.43s/it]

...

GCN loss on unlabled data: 1.841745376586914
GCN acc on unlabled data: 0.2849802371541502
attack loss: 1.8324589729309082


Perturbing graph: 100%|█████████▉| 2393/2394 [32:18<00:00,  2.09it/s]

GCN loss on unlabled data: 1.8636735677719116
GCN acc on unlabled data: 0.27628458498023717
attack loss: 1.8504986763000488


Perturbing graph: 100%|██████████| 2394/2394 [32:19<00:00,  1.23it/s]
Processing...
Done!


=== training GAT model ===
Epoch 0, training loss: 1.9457263946533203
Epoch 10, training loss: 1.6441421508789062
Epoch 20, training loss: 1.3560419082641602
Epoch 30, training loss: 1.1122709512710571
Epoch 40, training loss: 0.9371834993362427
Epoch 50, training loss: 0.8058394193649292
Epoch 60, training loss: 0.6383964419364929
Epoch 70, training loss: 0.6261339783668518
Epoch 80, training loss: 0.6173667907714844
Epoch 90, training loss: 0.5261088609695435
Epoch 100, training loss: 0.5753949880599976
Epoch 110, training loss: 0.4732600152492523
Epoch 120, training loss: 0.5086972117424011
Epoch 130, training loss: 0.448129802942276
Epoch 140, training loss: 0.4189615547657013
=== early stopping at 145, loss_val = 0.9458887577056885 ===
Test set results: loss= 0.9138 accuracy= 0.7091
accuracy:  0.7090747330960854
benchmark change:  -0.14145907473309616


## GSAINT

In [13]:
# Setup Surrogate model
surrogate_saint = GSAINT(nfeat=features.shape[1], nhid=64, nclass=labels.max().item()+1, dropout=0, with_bias=True, device=device).to(device)
surrogate_saint.fit(data, patience=100, verbose=True)

Processing...
Done!
Compute GraphSAINT normalization: : 289672it [00:00, 1892763.45it/s]                          


=== training GSAINT model ===
Epoch 0, training loss: 0.15944884717464447
Epoch 10, training loss: 0.0030045376624912024
Epoch 20, training loss: 0.002383779501542449
Epoch 30, training loss: 0.040824174880981445
Epoch 40, training loss: 0.0026719977613538504
Epoch 50, training loss: 0.002355232834815979
Epoch 60, training loss: 0.0022968854755163193
Epoch 70, training loss: 0.00236640777438879
Epoch 80, training loss: 0.0023276093415915966
Epoch 90, training loss: 0.0023281911853700876
Epoch 100, training loss: 0.002275155857205391
=== early stopping at 109, loss_val = 0.5772396922111511 ===


In [14]:
preds=surrogate_saint.predict()
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

Test Accuracy: 0.8509786476868327


In [15]:
benchmark_clean = test_accuracy

In [16]:
from copy import deepcopy

gsaint_results = []

for ptb in ptb_rates:
  torch.cuda.empty_cache()
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate_saint, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  data_copied = deepcopy(data)
  data_copied.adj = modified_adj
  atk_model = GSAINT(nfeat=features.shape[1], nhid=64, nclass=labels.max().item()+1, dropout=0, with_bias=True, device=device).to(device)
  atk_model.fit(data_copied, patience=100, verbose=True)

  atk_acc = atk_model.test()
  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  gsaint_results.append(atk_acc - benchmark_clean)


Perturbing graph:   0%|          | 0/399 [00:00<?, ?it/s]

GCN loss on unlabled data: 0.9595130085945129
GCN acc on unlabled data: 0.8027667984189724
attack loss: 0.9058895707130432


Perturbing graph:   0%|          | 1/399 [00:00<03:24,  1.95it/s]

GCN loss on unlabled data: 0.9845173358917236
GCN acc on unlabled data: 0.7964426877470355
attack loss: 0.9288957118988037


Perturbing graph:   1%|          | 2/399 [00:00<03:15,  2.03it/s]

...

GCN loss on unlabled data: 2.334519624710083
GCN acc on unlabled data: 0.38300395256916997
attack loss: 2.371286153793335


Perturbing graph: 100%|█████████▉| 2393/2394 [20:02<00:00,  1.98it/s]

GCN loss on unlabled data: 2.2998735904693604
GCN acc on unlabled data: 0.37075098814229246
attack loss: 2.3353967666625977


Perturbing graph: 100%|██████████| 2394/2394 [20:02<00:00,  1.99it/s]
Processing...
Done!
Compute GraphSAINT normalization: : 291351it [00:00, 1826967.43it/s]                          


=== training GSAINT model ===
Epoch 0, training loss: 0.16813234984874725
Epoch 10, training loss: 0.002887130482122302
Epoch 20, training loss: 0.0027785301208496094
Epoch 30, training loss: 0.008932026103138924
Epoch 40, training loss: 0.002607308328151703
Epoch 50, training loss: 0.002658864948898554
Epoch 60, training loss: 0.0026026046834886074
Epoch 70, training loss: 0.0026523927226662636
Epoch 80, training loss: 0.0026385148521512747
Epoch 90, training loss: 0.0026392403524369
Epoch 100, training loss: 0.0025625512935221195
Epoch 110, training loss: 0.0026024477556347847
Epoch 120, training loss: 0.0026115544606000185
Epoch 130, training loss: 0.0026153922080993652
=== early stopping at 130, loss_val = 1.274059534072876 ===
accuracy:  0.5484875444839857
benchmark change:  -0.302491103202847


## GSAGE

In [ ]:
# Setup Surrogate model
surrogate_sage = GSAGE(nfeat=features.shape[1], nhid=64, nclass=labels.max().item()+1, dropout=0, with_bias=True, device=device).to(device)
surrogate_sage.fit(data, patience=100, verbose=True)

In [ ]:
preds=surrogate_sage.predict()
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

In [ ]:
benchmark_clean = test_accuracy

In [ ]:
from copy import deepcopy

gsage_results = []

for ptb in ptb_rates:
  torch.cuda.empty_cache()
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate_sage, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  data_copied = deepcopy(data)
  data_copied.adj = modified_adj
  atk_model = GSAGE(nfeat=features.shape[1], nhid=64, nclass=labels.max().item()+1, dropout=0, with_bias=True, device=device).to(device)
  atk_model.fit(data_copied, patience=100, verbose=True)

  atk_acc = atk_model.test()
  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  gsage_results.append(atk_acc - benchmark_clean)


## GCN

In [ ]:
# Setup Surrogate model
surrogate = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=16, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
surrogate.fit(features, adj, labels, idx_train, idx_val, patience=100, verbose=True)

In [ ]:
preds=surrogate.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total

print(f'Test Accuracy: {test_accuracy}')

In [ ]:
benchmark_clean = test_accuracy

In [ ]:
from copy import deepcopy

gcn_results = []

for ptb in ptb_rates:
  torch.cuda.empty_cache()
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  atk_model = GCN(nfeat=features.shape[1], nclass=labels.max().item()+1,
                nhid=16, dropout=0, with_relu=True, with_bias=True, device=device).to(device)
  atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

  atk_acc = atk_model.test(idx_test)
  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  gcn_results.append(atk_acc - benchmark_clean)


In [ ]:
gcn_results

## GCNJaccard

In [ ]:
surrogate1 = GCNJaccard(nfeat=features.shape[1],
          nhid=16,
          nclass=labels.max().item() + 1,
          dropout=0.5, device=device).to(device)
surrogate1.fit(features, adj, labels, idx_train, idx_val, threshold=0.03)

In [ ]:
preds=surrogate1.predict(features,adj)
_, predicted_classes = torch.max(preds, 1)  # Returns values, indices; we need indices

# Step 3: Subset for test nodes
test_preds = predicted_classes[idx_test].cpu().numpy()
test_labels = labels[idx_test]

# Step 4: Calculate accuracy
correct = (test_preds == test_labels).sum().item()  # Count correct predictions
total = idx_test.shape[0]  # Total number of test nodes
test_accuracy = correct / total
benchmark_clean = test_accuracy

print(f'Test Accuracy: {test_accuracy}')

In [ ]:
from copy import deepcopy

# ptb_rates = [0.1, 0.2, 0.3, 0.4, 0.5]
jaccard_results = []

for ptb in ptb_rates:
  budget = int(ptb * (adj.todense().sum() // 2))
  # Setup Attack Model
  model = Metattack(surrogate1, nnodes=adj.shape[0], feature_shape=features.shape,
          attack_structure=True, attack_features=False, device=device, lambda_=0).to(device)
  # Attack
  model.attack(features, adj, labels, idx_train, idx_unlabeled, n_perturbations=budget, ll_constraint=False)
  modified_adj = model.modified_adj # modified_adj is a torch.tensor
  modified_adj = modified_adj.cpu().numpy()
  modified_adj = csr_matrix(modified_adj)

  perturbed_adj = deepcopy(modified_adj)
  atk_model = GCNJaccard(nfeat=features.shape[1],
            nhid=16,
            nclass=labels.max().item() + 1,
            dropout=0.5, device=device).to(device)
  atk_model.fit(features, modified_adj, labels, idx_train, idx_val, patience=100, verbose=True)

  atk_acc = atk_model.test(idx_test)

  print("accuracy: ", atk_acc)
  print("benchmark change: ", atk_acc - benchmark_clean)
  jaccard_results.append(atk_acc - benchmark_clean)


## Plotting

In [ ]:
# Plotting
plt.plot(ptb_rates, gcn_results, label="GCN")
plt.plot(ptb_rates, jaccard_results, label="Jaccard")
plt.plot(ptb_rates, gat_results, label="GAT")
plt.plot(ptb_rates, gsage_results, label="GSAGE")
plt.plot(ptb_rates, gsaint_results, label="GSAINT")

plt.xlabel('PTB Rates')
plt.ylabel('Change in Accuracy (abs. value)')
plt.title(f'Attacks on {dataset} Dataset')
plt.legend()

# Show plot
plt.show()

output_path = f"./data/{dataset}_graph.png"
plt.savefig(output_path)